# Giải mã Dữ liệu và Kỹ thuật Đặc trưng (Feature Engineering)
Notebook này tiếp tục quy trình chuẩn bị dữ liệu bằng cách giải mã các biến phân loại (categorical variables) từ dạng mã số sang chuỗi văn bản có ý nghĩa, tính toán thêm đặc trưng mới (tuổi xe) và loại bỏ các cột không dùng đến cho mô hình.
## Data Dictionary (Từ điển dữ liệu sau giải mã)
- gearbox (Hộp số): 1: Số sàn, 2: Số tự động
- fuel (Nhiên liệu): 1: Xăng, 2: Dầu, 3: Hybrid, 4: Điện
- cartype (Kiểu dáng): 1: Sedan, 2: SUV, 3: Hatchback, 4: Crossover, 5: MPV, 6: Bán tải, 7: Van/Minivan
- carcolor (Màu sắc): 1: Trắng, 2: Đen, 3: Bạc, 4: Đỏ, 5: Xám, 6: Xanh, 7: Nâu, 8: Vàng
- carorigin (Xuất xứ): 1: Lắp ráp trong nước, 2: Nhập khẩu, 3: Nhập Thái Lan, 4: Nhập Indonesia
- carseats (Số chỗ ngồi): 1: 4-5 chỗ, 2: 6-8 chỗ, 3: 2-3 chỗ (Bán tải), 4: 9-16 chỗ
- car_age (Tuổi xe): Tính bằng: Năm hiện tại (2026) - Năm sản xuất (mfdate)

In [2]:
import pandas as pd

In [3]:
# 1. ĐỌC DỮ LIỆU TỪ BƯỚC TRƯỚC
df = pd.read_csv('cars_cleaned.csv')
print(f"Kích thước dữ liệu ban đầu: {df.shape}")


Kích thước dữ liệu ban đầu: (13614, 38)


### Bước 1: Loại bỏ các cột không cần thiết
Các cột chứa liên kết hình ảnh (images, image) và nguồn (source_url) không có giá trị cho các mô hình học máy dạng bảng (Tabular Machine Learning). Việc loại bỏ chúng giúp giảm dung lượng bộ nhớ và tăng tốc độ xử lý.


In [4]:
# Sử dụng errors='ignore' để code không báo lỗi nếu file trước đó đã lỡ xóa các cột này
cols_to_drop = ['images', 'image', 'source_url']
df = df.drop(columns=cols_to_drop, errors='ignore')
print("Đã loại bỏ các cột hình ảnh và URL.")


Đã loại bỏ các cột hình ảnh và URL.


### Bước 2: Giải mã các cột mã số (Decoding)
Chuyển đổi các con số vô tri thành các nhãn phân loại (labels) rõ ràng. Điều này không chỉ giúp dữ liệu dễ đọc hơn khi làm EDA (Phân tích khám phá), mà còn cho phép thuật toán dễ dàng thực hiện One-Hot Encoding sau này.


In [8]:
# Định nghĩa các bộ từ điển giải mã
gearbox_map = {1: 'Số sàn', 2: 'Số tự động', 3: 'Số vô cấp CVT'}
fuel_map = {1: 'Xăng', 2: 'Dầu', 3: 'Hybrid', 4: 'Điện'}
cartype_map = {1: 'Sedan', 2: 'SUV', 3: 'Hatchback', 4: 'Crossover', 5: 'MPV', 6: 'Bán tải', 7: 'Van/Minivan'}
carcolor_map = {1: 'Trắng', 2: 'Đen', 3: 'Bạc', 4: 'Đỏ', 5: 'Xám', 6: 'Xanh', 7: 'Nâu', 8: 'Vàng'}
carorigin_map = {1: 'Lắp ráp trong nước', 2: 'Nhập khẩu', 3: 'Nhập Thái Lan', 4: 'Nhập Indonesia'}
carseats_map = {1: '4-5 chỗ', 2: '6-8 chỗ', 3: '2-3 chỗ (Bán tải)', 4: '9-16 chỗ'}
# Áp dụng mapping vào DataFrame (dùng map hoặc replace)
if 'gearbox' in df.columns:
    df['gearbox'] = df['gearbox'].map(gearbox_map).fillna(df['gearbox'])
if 'fuel' in df.columns:
    df['fuel'] = df['fuel'].map(fuel_map).fillna(df['fuel'])
if 'cartype' in df.columns:
    df['cartype'] = df['cartype'].map(cartype_map).fillna(df['cartype'])
if 'carcolor' in df.columns:
    df['carcolor'] = df['carcolor'].map(carcolor_map).fillna(df['carcolor'])
if 'carorigin' in df.columns:
    df['carorigin'] = df['carorigin'].map(carorigin_map).fillna(df['carorigin'])
if 'carseats' in df.columns:
    df['carseats'] = df['carseats'].map(carseats_map).fillna(df['carseats'])
print("Giải mã hoàn tất!")

Giải mã hoàn tất!


### Bước 3: Feature Engineering - Tính tuổi xe
Thay vì để thuật toán tự học sự liên quan của năm sản xuất (mfdate), chúng ta sẽ tạo ra một đặc trưng mới là car_age (Tuổi xe). Đây là một chỉ số tuyến tính và mạnh mẽ hơn rất nhiều trong việc phản ánh độ hao mòn và sự mất giá của phương tiện.

In [9]:
CURRENT_YEAR = 2026
if 'mfdate' in df.columns:
    # Ép kiểu năm sản xuất về numeric trước khi tính toán
    df['mfdate'] = pd.to_numeric(df['mfdate'], errors='coerce')
    # Tính tuổi xe, những xe chưa có mfdate sẽ được tính là NaN, sau đó fill bằng 0 hoặc trung vị
    df['car_age'] = CURRENT_YEAR - df['mfdate']
    # Xử lý các giá trị âm (trường hợp xe đời mới của năm tiếp theo)
    df.loc[df['car_age'] < 0, 'car_age'] = 0
    print("Đã thêm cột 'car_age' thành công.")
else:
    print("Không tìm thấy cột 'mfdate' trong dữ liệu.")

Đã thêm cột 'car_age' thành công.


### Bước 4: Xuất bộ dữ liệu sạch cuối cùng
Kiểm tra tổng quan dữ liệu lần cuối và lưu thành file CSV mới để sẵn sàng cho bước Trực quan hóa (EDA) hoặc Huấn luyện mô hình (Model Training).

In [11]:
print("--- Xem trước 5 dòng dữ liệu đầu tiên ---")
display(df.head())
output_filename = 'cars_cleaned_final.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"\nHoàn thành! Đã lưu bộ dữ liệu sạch và được giải mã vào file: {output_filename}")

--- Xem trước 5 dòng dữ liệu đầu tiên ---


,list_id,ad_id,subject,price,price_string,category,category_name,condition_ad,condition_ad_name,carbrand,...,list_time,orig_list_time,number_of_images,type,status,fetched_at,price_million,flag_chinh_chu,flag_bao_duong_hang,car_age
0,94613937,131588155,ISUZU DMAX NHẬP THÁI SỐ TỰ ĐỘNG SIÊU KENG RẤT ĐẸP,475000000,475.000.000 đ,2010,Ô tô,1,Đã sử dụng,13,...,1786860877000,NaN,12,s,active,2026-09-14T06:19:28.426335+00:00,475.0,0,0,12
1,101546338,140022407,Ranger XLS AT,615000000,615.000.000 đ,2010,Ô tô,1,Đã sử dụng,3,...,1787635662939,1.786291e+12,15,s,active,2026-09-14T06:22:18.715141+00:00,615.0,0,0,1
2,101725648,140238757,terano ll máy dầu 7 chỗ diezen 2 cầu tubo xe đẹp,140000000,140.000.000 đ,2010,Ô tô,1,Đã sử dụng,19,...,1785372154000,NaN,8,s,active,2026-09-14T06:14:39.741275+00:00,140.0,0,0,24
3,103030971,141801546,BMW X6 2010 Full Đồ Chơi Của Hãng,638000000,638.000.000 đ,2010,Ô tô,1,Đã sử dụng,9,...,1788411834744,1.785025e+12,10,s,active,2026-09-14T06:22:05.016826+00:00,638.0,0,0,16
4,109039668,148969479,Toyota Fortuner 4x2 2016 một chủ,439000000,439.000.000 đ,2010,Ô tô,1,Đã sử dụng,2,...,1787024581986,1.785165e+12,20,s,active,2026-09-14T06:19:24.205751+00:00,439.0,1,0,10



Hoàn thành! Đã lưu bộ dữ liệu sạch và được giải mã vào file: cars_cleaned_final.csv
